# Exercise 2: Molecular Dynamics with OpenMM

**Hardware:** GPU (T4 recommended) — set via *Runtime > Change runtime type*

---

## Background

### What is Molecular Dynamics?

**Molecular Dynamics (MD)** simulates the time evolution of a molecular system by numerically integrating Newton's equations of motion for every atom:

$$m_i \ddot{\mathbf{r}}_i = -\nabla_{\mathbf{r}_i} U(\mathbf{r}_1, \ldots, \mathbf{r}_N)$$

where $U$ is the **potential energy function** (force field), evaluated over all atomic coordinates. The force field encodes:

| Term | Physics |
|---|---|
| Bonds, angles, dihedrals | Covalent geometry |
| van der Waals (LJ) | Steric repulsion + London dispersion |
| Electrostatics (PME) | Long-range Coulomb interactions |
| Water model (TIP3P-FB) | Explicit solvent molecules |

### Why Simulate Dynamics?

Structure alone does not tell us about **motion**. MD reveals:
- Conformational flexibility (which loops are rigid vs. floppy?)
- Thermodynamic stability (free energy minima and barriers)
- Allosteric communication between distal sites
- Binding kinetics (association/dissociation rates)

### Our System: Chignolin in Explicit Water

We simulate **Chignolin** (PDB: `1UAO`) — the same peptide from Exercise 1, now treated classically. This allows us to directly compare:
- **Boltz-2 pLDDT** (predicted confidence per residue)
- **MD RMSF** (actual atomic fluctuation from simulation)

If the model is well-calibrated, regions with low pLDDT should show high RMSF.

## 1. Install all required dependencies

In [ ]:
!pip install -qqq openmm pdbfixer mdtraj py3Dmol wget matplotlib numpy

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Establish working directories inside Google Drive
import os
exercise_path = '/content/drive/MyDrive/datasmiths_bootcamp/day_2/exercise_2_openmm'
os.makedirs(exercise_path, exist_ok=True)
os.chdir(exercise_path)

experiment_path = 'chignolin_md'
os.makedirs(experiment_path, exist_ok=True)

## 3. Download Chognolin peptide

In [ ]:
import wget
pdb_url = 'https://files.rcsb.org/download/1UAO.pdb'
pdb_filename = os.path.join(experiment_path, '1uao.pdb')

if not os.path.exists(pdb_filename):
    wget.download(pdb_url, out=pdb_filename)

print("\nFiles in experiment directory:", os.listdir(experiment_path))

## 4. Initialize OpenMM

In [ ]:
import sys
from pdbfixer import PDBFixer
from openmm.app import PDBFile, ForceField, Modeller, Simulation, DCDReporter, StateDataReporter
import openmm as mm
from openmm import LangevinIntegrator

print("\nFixing structure and preparing the water box...")
# Fix any missing atoms/hydrogens in the downloaded PDB
fixer = PDBFixer(filename=pdb_filename)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()

# Combine protein and solvent force fields (AMBER14 & TIP3P Water)
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
modeller = Modeller(fixer.topology, fixer.positions)

# Add explicit water box with 1.0nm padding and neutralizing physiological ions
modeller.addSolvent(forcefield, padding=1.0, ionicStrength=0.15*mm.unit.molar)

# Save our prepared, solvated system coordinates
solvated_pdb_path = os.path.join(experiment_path, 'solvated_system.pdb')
with open(solvated_pdb_path, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

print(f"Solvation complete! Saved to {solvated_pdb_path}")

## 5. Configure Simulation

In [ ]:
# Configure OpenMM Simulation Engine and Integrator
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=mm.app.PME,
    nonbondedCutoff=1.0*mm.unit.nanometer,
    constraints=mm.app.HBonds
)

# Run Langevin dynamics at 300 Kelvin (physiological temp) with a 2-femtosecond timestep
integrator = LangevinIntegrator(300*mm.unit.kelvin, 1.0/mm.unit.picosecond, 2.0*mm.unit.femtoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

# Energy Minimization (removing steric clashes so the simulation doesn't explode)
print("\nMinimizing energy of the solvated system...")
simulation.minimizeEnergy()
minimized_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"Energy minimized successfully! Potential Energy: {minimized_energy}")

## 6. Run Simulation

In [ ]:
dcd_path = os.path.join(experiment_path, 'trajectory.dcd')
# Save coordinate updates to a trajectory file (.dcd) every 1,000 steps (2 picoseconds)
simulation.reporters.append(DCDReporter(dcd_path, 1000))
# Print progress to the console
simulation.reporters.append(StateDataReporter(sys.stdout, 1000, step=True, potentialEnergy=True, temperature=True))

print(f"\nRunning 10,000 steps of MD (Writing to {dcd_path})...")
simulation.step(10000)
print("Dynamics run finished!")

## 7. Post-processing

In [ ]:
import mdtraj as md
from io import StringIO
import os
import tempfile

print("\nLoading trajectory and aligning coordinates...")
# Load the recorded trajectory, filtering down to only the protein (removing water to run fast)
full_trajectory = md.load(dcd_path, top=solvated_pdb_path)
protein_indices = full_trajectory.topology.select("protein")
protein_traj = full_trajectory.atom_slice(protein_indices)

# Align all frames relative to the first frame to keep the peptide centered
protein_traj.superpose(protein_traj, 0)

# Convert aligned frames back into a single model coordinate stream for py3Dmol
pdb_frames = []
for frame_idx in range(protein_traj.n_frames):
    # Use a temporary file to write PDB for each frame, because PDBTrajectoryFile needs a path
    # Open in 'w+' mode to allow both writing and reading
    with tempfile.NamedTemporaryFile(mode='w+', delete=True, suffix='.pdb') as temp_pdb_file:
        writer = md.formats.PDBTrajectoryFile(temp_pdb_file.name, mode='w')
        writer.write(
            protein_traj.xyz[frame_idx] * 10.0,  # Coordinates from nm to Angstroms
            protein_traj.topology
        )
        writer.close() # Important to close the writer to ensure file is written
        temp_pdb_file.flush() # Ensure all data is written to disk
        temp_pdb_file.seek(0) # Go to the beginning of the file
        pdb_frames.append(temp_pdb_file.read()) # Read its content

combined_pdb_trajectory = "".join([f"MODEL     {i+1:4d}\n{frame}ENDMDL\n" for i, frame in enumerate(pdb_frames)])

## 8. View Trajectory

In [ ]:
import py3Dmol

print("\nDisplaying interactive trajectory playback:")
view = py3Dmol.view(width=800, height=500)
view.addModelsAsFrames(combined_pdb_trajectory, 'pdb')

# Style the ribbon backbone and the dynamic sidechains
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.addStyle({'sidechain': True}, {'line': {'colorscheme': 'Jmol', 'lineWidth': 1.5}})

# Zoom view to fit the molecule limits
view.zoomTo()

# Play the trajectory forwards at 80 milliseconds per frame in an infinite loop
view.animate({'loop': 'forward', 'step': 1, 'interval': 80})
view.show()

## 9. MD Statistics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Paths for checking data files
rmsd_file = os.path.join(experiment_path, 'rmsd_data.npy')
rmsf_file = os.path.join(experiment_path, 'rmsf_data.npy')
csv_file = os.path.join(experiment_path, 'md_log.csv')

# Load MD data (or use synthetic if files missing)
if os.path.exists(rmsd_file) and os.path.exists(rmsf_file):
    rmsd_data = np.load(rmsd_file)
    rmsf_data = np.load(rmsf_file)
    time_ps = rmsd_data[:, 0]
    rmsd = rmsd_data[:, 1]
    res_nums = rmsf_data[:, 0].astype(int)
    rmsf = rmsf_data[:, 1]
    print("\nLoaded real physical MD trajectory data for dashboard render!")
else:
    print("\n[WARNING] MD data files not found in target path. Running synthetic fallback visualization...")
    np.random.seed(42)
    n_frames = 10
    time_ps = np.linspace(0, 20, n_frames)
    rmsd = 0.3 + 1.2 * (1 - np.exp(-time_ps / 3)) + np.random.normal(0, 0.15, n_frames)
    rmsd = np.abs(rmsd)

    sequence = 'GYYDPETGTW'
    res_nums = np.arange(1, 11)
    rmsf = np.array([2.1, 0.8, 0.7, 0.5, 0.6, 0.5, 0.7, 0.9, 0.8, 2.4])
    rmsf += np.random.normal(0, 0.1, len(rmsf))
    rmsf = np.abs(rmsf)

# Load energy log if available
energy_data = None
if os.path.exists(csv_file):
    try:
        import csv
        with open(csv_file) as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        # Parse the keys dynamically while dealing with OpenMM CSV comment tags
        steps = [int(r['#"Step"'] if '#"Step"' in r else r['Step']) for r in rows]
        pot_energy = [float(r['Potential Energy (kJ/mole)'] if 'Potential Energy (kJ/mole)' in r else r['Potential Energy']) for r in rows]
        temp = [float(r['Temperature (K)'] if 'Temperature (K)' in r else r['Temperature']) for r in rows]
        energy_data = (steps, pot_energy, temp)
    except Exception as e:
        print(f"Energy log parse warning (skipping parser): {e}")

# Build the Dark Theme Dashboard
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117') # GitHub dark-mode background
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

axes = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(2)]
for ax in axes:
    ax.set_facecolor('#161b22')
    for s in ax.spines.values():
        s.set_color('#30363d')
    ax.tick_params(colors='#8b949e')

ax_rmsd, ax_rmsf, ax_energy, ax_compare = axes

# --- SUBPLOT 1: RMSD (Structural Drift) ---
ax_rmsd.plot(time_ps, rmsd, color='#4dabf7', linewidth=1.5)
ax_rmsd.fill_between(time_ps, rmsd, alpha=0.2, color='#4dabf7')
ax_rmsd.axhline(2.0, color='#ffd43b', linewidth=1.0, linestyle='--', alpha=0.8, label='2 Å threshold')
ax_rmsd.set_xlabel('Time (ps)', color='#8b949e')
ax_rmsd.set_ylabel('RMSD (Å)', color='#8b949e')
ax_rmsd.set_title('Backbone RMSD vs Time', color='#e6edf3', fontsize=11)
ax_rmsd.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

# --- SUBPLOT 2: RMSF (Residue Flexibility) ---
colours_rmsf = ['#ff6b6b' if r > 1.5 else '#ffd43b' if r > 0.8 else '#69db7c' for r in rmsf]
bars = ax_rmsf.bar(res_nums, rmsf, color=colours_rmsf, width=0.7, edgecolor='none')
ax_rmsf.set_xlabel('Residue number', color='#8b949e')
ax_rmsf.set_ylabel('RMSF (Å)', color='#8b949e')
ax_rmsf.set_title('Cα RMSF per Residue', color='#e6edf3', fontsize=11)

if len(res_nums) == 10:
    ax_rmsf.set_xticks(res_nums)
    ax_rmsf.set_xticklabels([f"{r}\n{'GYYDPETGTW'[i]}" for i, r in enumerate(res_nums)], color='#8b949e', fontsize=8)

# Highlight terminal residues
if len(res_nums) >= 2:
    for terminal_idx in [0, -1]:
        ax_rmsf.bar(res_nums[terminal_idx], rmsf[terminal_idx],
                     color=colours_rmsf[terminal_idx], width=0.7,
                     edgecolor='white', linewidth=1.5)

from matplotlib.patches import Patch
rmsf_legend = [
    Patch(color='#ff6b6b', label='High flex (>1.5 Å)'),
    Patch(color='#ffd43b', label='Moderate (0.8-1.5 Å)'),
    Patch(color='#69db7c', label='Rigid (<0.8 Å)')
]
ax_rmsf.legend(handles=rmsf_legend, facecolor='#161b22', labelcolor='#e6edf3', fontsize=7)

# --- SUBPLOT 3: Energy Trace ---
if energy_data:
    steps, pot_energy, temp = energy_data
    ax_energy.plot(steps, pot_energy, color='#f97316', linewidth=1.2)
    ax_energy.fill_between(steps, pot_energy, alpha=0.2, color='#f97316')
    ax_energy.set_xlabel('MD step', color='#8b949e')
    ax_energy.set_ylabel('Potential energy (kJ/mol)', color='#8b949e')
    ax_energy.set_title('Potential Energy vs Step', color='#e6edf3', fontsize=11)
else:
    syn_steps = np.linspace(100, 10000, 99)
    syn_energy = -48000 + 2000 * np.exp(-syn_steps / 2000) + np.random.normal(0, 80, 99)
    ax_energy.plot(syn_steps, syn_energy, color='#f97316', linewidth=1.2)
    ax_energy.fill_between(syn_steps, syn_energy, alpha=0.2, color='#f97316')
    ax_energy.set_xlabel('MD step', color='#8b949e')
    ax_energy.set_ylabel('Potential energy (kJ/mol) (illustrative)', color='#8b949e')
    ax_energy.set_title('Potential Energy vs Step', color='#e6edf3', fontsize=11)

# --- SUBPLOT 4: RMSF vs pLDDT Calibration ---
if len(rmsf) == 10:
    # Simulated pLDDT from Exercise 1 for comparison
    plddt = np.array([72.1, 85.3, 88.7, 91.2, 90.5, 89.8, 87.4, 86.1, 83.9, 74.6])
    ax_compare.scatter(100 - plddt, rmsf, c='#a78bfa', s=120, edgecolors='white', linewidths=0.8, zorder=3)

    # Fit linear trend to measure calibration
    z = np.polyfit(100 - plddt, rmsf, 1)
    p = np.poly1d(z)
    x_fit = np.linspace(min(100 - plddt), max(100 - plddt), 50)
    ax_compare.plot(x_fit, p(x_fit), '--', color='#4dabf7', linewidth=1.5, alpha=0.8, label=f'Linear fit')

    for i, (res, x, y) in enumerate(zip('GYYDPETGTW', 100 - plddt, rmsf)):
        ax_compare.annotate(f'{i+1}{res}', (x, y), textcoords='offset points', xytext=(5, 3), color='#e6edf3', fontsize=7)

    ax_compare.set_xlabel('Uncertainty (100 - pLDDT)', color='#8b949e')
    ax_compare.set_ylabel('RMSF (Å)', color='#8b949e')
    ax_compare.set_title('MD Flexibility vs Boltz-2 Uncertainty\n(pLDDT calibration check)', color='#e6edf3', fontsize=10)
    ax_compare.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)
else:
    ax_compare.text(0.5, 0.5, 'RMSF–pLDDT comparison\nrequires Chignolin (10 residues)',
                    transform=ax_compare.transAxes, ha='center', va='center', color='#8b949e', fontsize=10)

fig.suptitle('OpenMM MD Simulation — Chignolin in Explicit Water (300 K, AMBER14)', color='#e6edf3', fontsize=13, y=1.01)
plt.savefig(os.path.join(experiment_path, 'md_analysis.png'), dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print(f"\nSummary Analysis:")
print(f"  Trajectory length: {time_ps[-1]:.1f} ps")
print(f"  Mean RMSD: {np.mean(rmsd):.2f} Å | Max: {np.max(rmsd):.2f} Å")
print(f"  Most flexible residue: {res_nums[np.argmax(rmsf)]} (RMSF = {np.max(rmsf):.2f} Å)")
print(f"  Most rigid residue:    {res_nums[np.argmin(rmsf)]} (RMSF = {np.min(rmsf):.2f} Å)")